<a href="https://colab.research.google.com/github/francesc0patti/qml-project/blob/main/Settimana1_Intro_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Importiamo le librerie fondamentali per manipolare dati e grafici
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Importiamo la libreria scikit-learn che contiene i dataset e i modelli classici
from sklearn import datasets

# 1. Carichiamo il dataset Iris
iris = datasets.load_iris()

# 2. Mettiamo i dati in una "tabella" Pandas per renderli leggibili (simile a Excel)
df = pd.DataFrame(data=iris.data, columns=iris.feature_names)

# Aggiungiamo la colonna con le "etichette" (le specie di fiori che dovremo imparare a prevedere)
df['target'] = iris.target

# 3. Mostriamo le prime 5 righe della nostra tabella per verificare che tutto funzioni
df.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


In [4]:
!pip install pennylane

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 93.3 MB/s eta 0:00:00
  Attempting uninstall: autograd
    Found existing installation: autograd 1.9.1
    Uninstalling autograd-1.9.1:
      Successfully uninstalled autograd-1.9.1


In [3]:
import pennylane as qml
from pennylane import numpy as np
from sklearn import datasets
from sklearn.preprocessing import StandardScaler

# ==========================================
# 1. PREPARAZIONE DATI
# ==========================================
iris = datasets.load_iris()
X = iris.data
Y = iris.target

# Il One-Hot Encoding ora ha 3 colonne (perché le nostre classi reali sono 3)
Y_one_hot = np.zeros((len(Y), 3))
for i in range(len(Y)):
    Y_one_hot[i, Y[i]] = 1.0

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


# ==========================================
# 2. IL CIRCUITO A 16 STATI
# ==========================================
dev = qml.device("default.qubit", wires=4)

@qml.qnode(dev)
def circuito_completo(dati_input, pesi):
    qml.AngleEmbedding(features=dati_input, wires=range(4), rotation='Y')
    qml.BasicEntanglerLayers(weights=pesi, wires=range(4))

    # LA TUA IDEA: Misuriamo TUTTI e 4 i Qubit.
    # Questo restituisce un array di 16 probabilità (da |0000> a |1111>)
    return qml.probs(wires=range(4))


# ==========================================
# 3. MAPPATURA RIDONDANTE (Error Mitigation)
# ==========================================
def aggrega_probabilita(probs_16):
    # Suddividiamo i 16 stati del circuito nei 3 "bacini" delle nostre specie
    # Usiamo np.sum per sommare le onde di probabilità

    prob_setosa = np.sum(probs_16[0:5])       # Stati da 0 a 4 (5 stati)
    prob_versicolor = np.sum(probs_16[5:10])  # Stati da 5 a 9 (5 stati)
    prob_virginica = np.sum(probs_16[10:16])  # Stati da 10 a 15 (6 stati)

    # Restituiamo le 3 probabilità consolidate tramite np.stack
    # (indispensabile per far calcolare le derivate a PennyLane)
    return np.stack([prob_setosa, prob_versicolor, prob_virginica])

def calcola_errore(pesi, X, Y_oh):
    previsioni_aggregate = []

    for x in X:
        # 1. Il circuito calcola le probabilità sui 16 stati
        probs_16 = circuito_completo(x, pesi)

        # 2. Intercettiamo le probabilità e le raggruppiamo
        probs_3 = aggrega_probabilita(probs_16)
        previsioni_aggregate.append(probs_3)

    previsioni_aggregate = np.stack(previsioni_aggregate)
    return np.mean((Y_oh - previsioni_aggregate) ** 2)


# ==========================================
# 4. CICLO DI ADDESTRAMENTO
# ==========================================
ottimizzatore = qml.AdamOptimizer(stepsize=0.02)
pesi_quantistici = np.random.uniform(low=-np.pi, high=np.pi, size=(12, 4))
pesi_quantistici = np.tensor(pesi_quantistici, requires_grad=True)

print("Inizio addestramento con Mappatura Ridondante (16 stati -> 3 Classi)...\n")

for epoca in range(300):
    pesi_quantistici, errore = ottimizzatore.step_and_cost(lambda p: calcola_errore(p, X_scaled, Y_one_hot), pesi_quantistici)
    print(f"Epoca {epoca+1:2d} | Errore: {errore:.4f}")

print("\nAddestramento completato! Il modello ora usa bacini di stati tolleranti al rumore.")

ModuleNotFoundError: No module named 'pennylane'

In [2]:
import pennylane as qml
from pennylane import numpy as np
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

np.random.seed(0)

# ==========================================
# 1. PREPARAZIONE DATI
# ==========================================
iris = datasets.load_iris()
X, Y = iris.data, iris.target

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.25, stratify=Y, random_state=0
)

# MinMaxScaler in [0, pi]: ogni feature occupa un intervallo di angoli univoco
# (fit solo sul train, per non "sbirciare" il test)
scaler = MinMaxScaler(feature_range=(0, np.pi))
X_train = np.array(scaler.fit_transform(X_train), requires_grad=False)
X_test = np.array(scaler.transform(X_test), requires_grad=False)
Y_train = np.array(Y_train, requires_grad=False)
Y_test = np.array(Y_test, requires_grad=False)

# ==========================================
# 2. CIRCUITO CON DATA RE-UPLOADING
# ==========================================
N_QUBIT = 4
N_LAYER = 6

dev = qml.device("default.qubit", wires=N_QUBIT)

@qml.qnode(dev)
def circuito(x, pesi, scala):
    # x ha shape (N, 4): PennyLane elabora tutto il batch in un colpo solo
    for l in range(N_LAYER):
        # i dati rientrano a ogni layer, con una scala allenabile per qubit
        for i in range(N_QUBIT):
            qml.RY(scala[l, i] * x[:, i], wires=i)
        # parte allenabile + entanglement
        qml.BasicEntanglerLayers(pesi[l:l + 1], wires=range(N_QUBIT))
    # 3 osservabili -> 3 logit (uno per specie)
    return [qml.expval(qml.PauliZ(i)) for i in range(3)]

# ==========================================
# 3. READOUT + COSTO (softmax + cross-entropy)
# ==========================================
def logits(x, pesi, scala, beta):
    out = np.stack(circuito(x, pesi, scala), axis=1)  # (N, 3), valori in [-1, 1]
    return beta * out  # beta allenabile: allarga il range per il softmax

def log_softmax(z):
    z = z - np.max(z, axis=1, keepdims=True)
    return z - np.log(np.sum(np.exp(z), axis=1, keepdims=True))

def costo(pesi, scala, beta, x, y):
    lp = log_softmax(logits(x, pesi, scala, beta))
    return -np.mean(lp[np.arange(len(y)), y])

def accuracy(pesi, scala, beta, x, y):
    pred = np.argmax(logits(x, pesi, scala, beta), axis=1)
    return float(np.mean(pred == y))

# ==========================================
# 4. ADDESTRAMENTO
# ==========================================
pesi = np.array(np.random.normal(0, 0.1, (N_LAYER, N_QUBIT)), requires_grad=True)
scala = np.array(1.0 + 0.1 * np.random.randn(N_LAYER, N_QUBIT), requires_grad=True)
beta = np.array(3.0, requires_grad=True)

opt = qml.AdamOptimizer(stepsize=0.05)

print("Inizio addestramento (re-uploading + expval + cross-entropy)...\n")
for epoca in range(300):
    (pesi, scala, beta, _, _), c = opt.step_and_cost(
        costo, pesi, scala, beta, X_train, Y_train
    )
    if epoca % 10 == 0 or epoca == 299:
        acc_tr = accuracy(pesi, scala, beta, X_train, Y_train)
        acc_te = accuracy(pesi, scala, beta, X_test, Y_test)
        print(f"Epoca {epoca+1:3d} | Costo: {c:.4f} | Acc train: {acc_tr:.2f} | Acc test: {acc_te:.2f}")

print("\nAddestramento completato.")

ModuleNotFoundError: No module named 'pennylane'

In [ ]:
np.save("pesi_iris_quantistici.npy", pesi)
np.save("scala_irisi_quantistici.npy", scala)
np.save("beta_iris_quantistici.npy", beta)

In [ ]:
pesi_caricati = np.load("pesi_iris_quantistici.npy")
scala_caricata = np.load("scala_irisi_quantistici.npy")
beta_caricata = np.load("beta_iris_quantistici.npy")

# PennyLane richiede che diciamo esplicitamente se questi pesi possono essere modificati
pesi_quantistici = np.tensor(pesi_caricati, requires_grad=True)

In [ ]:
# ==========================================
# 5. TEST E ACCURATEZZA
# ==========================================

def calcola_accuratezza(pesi, X, Y_reali):
    previsioni_corrette = 0

    for i in range(len(X)):
        # 1. Il circuito sputa le 16 probabilità
        probs_16 = circuito_completo(X[i], pesi)

        # 2. Le raggruppiamo nei 3 bacini (le 3 specie)
        probs_3 = aggrega_probabilita(probs_16)

        # 3. La IA decide: la classe con la probabilità più alta vince
        classe_predetta = np.argmax(probs_3)

        # 4. Controlliamo se ha indovinato
        if classe_predetta == Y_reali[i]:
            previsioni_corrette += 1

    accuratezza = (previsioni_corrette / len(X)) * 100
    return accuratezza

# Calcoliamo l'accuratezza finale
acc = calcola_accuratezza(pesi_quantistici, X_scaled, Y)
print(f"L'accuratezza finale del modello quantistico è: {acc:.2f}%")